In [208]:
import numpy as np
import math
from scipy.linalg import block_diag
from SVD import SVD
#from Retrieve_P import get_R_t_from_F
#from eight_point_agl import eight_point
from simulation.camera_model import get_K, get_camera_pose
from simulation.scene_generator import PointsGenerator
from simulation.projection import  project_points,filter_visible

# Verify 8 points algo

In [209]:
import numpy as np
from SVD import SVD
from normalization import normalize

def linear_eq(points1, points2):
    length = min(points1.shape[0], points2.shape[0])
    A = np.zeros((length, 9))
    for i in range(length):
        A[i, 0] = points2[i, 0]*points1[i, 0]
        A[i, 1] = points2[i, 0]*points1[i, 1]
        A[i, 2] = points2[i, 0]
        A[i, 3] = points2[i, 1]*points1[i, 0]
        A[i, 4] = points2[i, 1]*points1[i, 1]
        A[i, 5] = points2[i, 1]
        A[i, 6] = points1[i, 0]
        A[i, 7] = points1[i, 1]
        A[i, 8] = 1
    return A

def eight_point1(points1, points2):

    length = min(points1.shape[0], points2.shape[0])
    #1: Normalization
    T1 = normalize(points1)
    T2 = normalize(points2)
    norm1 = (T1 @ points1.T).T
    norm2 = (T2 @ points2.T).T

    #2: Find the fundamental matrix of the normalized points
    A_hat = linear_eq(norm1, norm2)
    U, Sigma, V = np.linalg.svd(A_hat)
    F_hat = V[8, :].reshape(3, 3)

    #3: Replace F_hat but Fprime_hat such that det Fprime_hat = 0
    U1, Sigma1, V1 = np.linalg.svd(F_hat)
    Sigma1[2] = 0
    Fprime_hat = U1@np.diag(Sigma1)@V1
    
    #4: Denormalize
    F = T2.T@Fprime_hat@T1

    return F

def get_R_t_from_F1(F, K):
    """
    Function to retrieve the rotational matrix and translational matrix 
    given the fundamental matrix, assuming P = [I|0] and P' = [R|t]
    Input:
        - F: Fundamental matrix (3x3, np.array)
        - K: Intrinsic matrix (3x3, np.array)
    Ouput:
        - t:  Translational matrix
        - R1: Rotational matrix (option 1)
        - R2: Rotational matrix (option 2)
    """

    E = K.T@F@K # Essential matrix
    
    W = np.array([
        [0, -1, 0],
        [1, 0, 0],
        [0, 0, 1]
    ])
    Z = np.array([
        [0, 1, 0],
        [-1, 0, 0],
        [0, 0, 0]
    ])
    U, Sigma, V = np.linalg.svd(E)
    print(f"Sigma = {Sigma}")
    #t = U[:, 2]
    t_2 = U@Z@U.T
    t = np.array([t_2[2,1], t_2[0,2], t_2[1,0]])
    R1 = U@W@V
    R2 = U@W.T@V
    if np.linalg.det(R1) < 0: R1 = -R1
    if np.linalg.det(R2) < 0: R2 = -R2
    return t, R1, R2

def rotation_matrix_to_euler(R):
    """
    Extracts (Roll, Pitch, Yaw) in radians from a 3x3 rotation matrix.
    Assumes ZYX rotation order.
    """
    # 1. Check if we are in Gimbal Lock (Pitch is exactly +/- 90 degrees)
    # We do this by checking if the length of the first two elements of the first column is 0
    sy = math.sqrt(R[0, 0] * R[0, 0] +  R[1, 0] * R[1, 0])
    singular = sy < 1e-6

    if not singular:
        # Normal extraction
        roll  = math.atan2(R[2, 1], R[2, 2])
        pitch = math.atan2(-R[2, 0], sy)
        yaw   = math.atan2(R[1, 0], R[0, 0])
    else:
        # Gimbal lock extraction (Pitch is +/- 90)
        # In this case, Roll and Yaw become mathematically combined, 
        # so we force Yaw to 0 and dump all the rotation into Roll.
        roll  = math.atan2(-R[1, 2], R[1, 1])
        pitch = math.atan2(-R[2, 0], sy)
        yaw   = 0

    # Returns angles in Radians (X, Y, Z)
    return np.array([roll, pitch, yaw])

# --- Example Usage ---
# If you want to see the angles in degrees, which is much easier to read:




In [210]:
# 1 Génération de la scène 
bounds = np.array([-5, 5, -5, 5, 3, 8])
pts3d = PointsGenerator(nbPoints=80, seed=42, bounds=bounds)  

# 2 Définition des caméras 
K = get_K()
R1, t1 = np.eye(3), np.zeros(3)
R2, t2 = get_camera_pose(rz=10, tx=4)

# 3 Projection 
px1, d1 = project_points(pts3d, K, R1, t1)
px2, d2 = project_points(pts3d, K, R2, t2)

# 4 Filtrage 
vis = filter_visible(px1, d1) & filter_visible(px2, d2)
print(f"Points visibles dans les deux caméras : {vis.sum()} / {pts3d.shape[1]}")

px1_vis = px1[:, vis]
px2_vis = px2[:, vis]
K_inv = np.linalg.inv(K)
points1 = np.vstack((px1, np.ones((1, px1.shape[1]))))
points2 = np.vstack((px2, np.ones((1, px2.shape[1]))))
F = eight_point1(points1.T, points2.T)
print(f"x'^TFx = {points2[:,0].T@F@points1[:,0]}")
t_1, R_1, R_2 = get_R_t_from_F1(F, K)
print(f"True value of t2 = {t2}")
print(f"Value of norm of t2 = {t2/np.linalg.norm(t2)}")
print(f"Value of estimated norm of t2 = {t_1}")
print(f"True value of R2 = \n{R2}")
print(f"Value of estimate of R2 = \n{R_1}")
print(f"Value of estimate of R2 = \n{R_2}")

Points visibles dans les deux caméras : 26 / 80
x'^TFx = 1.3322676295501878e-15
Sigma = [1.31955122e+00 1.12632340e+00 3.06514633e-17]
True value of t2 = [4. 0. 0.]
Value of norm of t2 = [1. 0. 0.]
Value of estimated norm of t2 = [ 1.00000000e+00  1.48731847e-15 -2.13580965e-15]
True value of R2 = 
[[ 0.98480775 -0.17364818  0.        ]
 [ 0.17364818  0.98480775  0.        ]
 [ 0.          0.          1.        ]]
Value of estimate of R2 = 
[[ 9.84807753e-01 -1.73648178e-01 -4.19645701e-15]
 [-1.73105439e-01 -9.81729732e-01 -7.90015248e-02]
 [ 1.37184708e-02  7.78013141e-02 -9.96874495e-01]]
Value of estimate of R2 = 
[[ 9.84807753e-01 -1.73648178e-01 -1.73189528e-16]
 [ 1.73105439e-01  9.81729732e-01  7.90015248e-02]
 [-1.37184708e-02 -7.78013141e-02  9.96874495e-01]]


In [211]:
# 1 Génération de la scène 
bounds = np.array([-5, 5, -5, 5, 3, 8])
pts3d = PointsGenerator(nbPoints=80, seed=42, bounds=bounds)  

# 2 Définition des caméras 
K = get_K()
R1, t1 = np.eye(3), np.zeros(3)
R2, t2 = get_camera_pose(rz=10, rx = 3, ry= 15, tx=4, ty = 9, tz= 2)

# 3 Projection 
px1, d1 = project_points(pts3d, K, R1, t1)
px2, d2 = project_points(pts3d, K, R2, t2)

# 4 Filtrage 
vis = filter_visible(px1, d1) & filter_visible(px2, d2)
print(f"Points visibles dans les deux caméras : {vis.sum()} / {pts3d.shape[1]}")
K_inv = np.linalg.inv(K)
px1_vis = px1[:, vis]
px2_vis = px2[:, vis]

points1 = np.vstack((px1_vis, np.ones((1, px1_vis.shape[1]))))
points2 = np.vstack((px2_vis, np.ones((1, px2_vis.shape[1]))))
points1[2,:] = np.ones((1, px1_vis.shape[1]))
points2[2,:] = np.ones((1, px1_vis.shape[1]))
F = eight_point1(points1.T, points2.T)
print(f"x'^TFx = {points2[:,0].T@F@points1[:,0]}")
t_1, R_1, R_2 = get_R_t_from_F1(F, K)

print(f"True value of t2 = {t2}")
print(f"Value of norm of t2 = {t2/np.linalg.norm(t2)}")
print(f"Value of estimated norm of t2 = {t_1}")
print(f"True value of R2 = \n{R2}")
print(f"Value of estimate of R2 = \n{R_1}")
print(f"Value of estimate of R2 = \n{R_2}")
angles_rad = rotation_matrix_to_euler(R_1)
angles_deg = np.degrees(angles_rad)

print(f"Roll  (X): {angles_deg[0]:.2f}°")
print(f"Pitch (Y): {angles_deg[1]:.2f}°")
print(f"Yaw   (Z): {angles_deg[2]:.2f}°")

Points visibles dans les deux caméras : 2 / 80
x'^TFx = 0.03839629163711468
Sigma = [3.89432292e+00 7.43873958e-01 2.48511830e-17]
True value of t2 = [4. 9. 2.]
Value of norm of t2 = [0.39801488 0.89553347 0.19900744]
Value of estimated norm of t2 = [-0.39095762  0.9204023  -0.00342807]
True value of R2 = 
[[ 0.95125124 -0.16007044  0.26362573]
 [ 0.16773126  0.98581027 -0.00665899]
 [-0.25881905  0.05055265  0.96460206]]
Value of estimate of R2 = 
[[-0.37385698  0.11924426 -0.91978898]
 [ 0.89918095 -0.19653065 -0.39095949]
 [-0.2273864  -0.97321966 -0.03374782]]
Value of estimate of R2 = 
[[-0.38815847  0.05603802  0.91988735]
 [ 0.89477506 -0.21612334  0.39072789]
 [ 0.22070475  0.97475661  0.03374868]]
Roll  (X): -91.99°
Pitch (Y): 13.14°
Yaw   (Z): 112.58°


# Check if biduagonalization is correct

In [212]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12]
])
print(A)
U, Sigma, V = np.linalg.svd(A)
S = np.vstack((np.diag(Sigma), np.ones((U.shape[0] - len(Sigma), np.diag(Sigma).shape[1]))))
print(U @ S @ V)

U, Sigma, V = SVD(A)
print(U @ Sigma @ V.T)

[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]
[[ 0.92192626  1.86808084  3.03737115]
 [ 3.75907793  4.59292028  6.1153209 ]
 [ 7.71606535  9.20991692  8.65724475]
 [ 9.60293045 10.32908196 12.1900632 ]]
[[ 1.00046705  2.37627287  2.66421893]
 [ 4.0002414   5.19448135  5.82644735]
 [ 7.00001575  8.01268982  8.98867577]
 [ 9.9997901  10.83089829 12.15090419]]


# Check if SVD is correct or not

In [213]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [10, 11, 12]
])
U, Sigma, V = SVD(A)
print(B)
print(U@Sigma@V.T)

NameError: name 'B' is not defined